In [30]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 10:47:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [3]:
dfe = spark.read.parquet(full_file_path).cache()
dfe.createOrReplaceTempView("e_table")

26/08/12 10:48:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [ ]:
# Event Structer

In [14]:
dfe.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: string (nullable = true)
 |-- device_type: string (nullable = true)



In [16]:
dfe.show(5,truncate=False, vertical=True)

-RECORD 0------------------------------------------------
 user_id          | 3dc19c4b-098c-4791-b993-93a22c544d97 
 video_id         | 33ea938f-667b-439a-8e70-99cf6d78007f 
 event_id         | 8c6973d5-0dae-42ad-906a-a6c57685d7d8 
 event_timestamp  | 2026-07-13T19:23:14.473538+00:00     
 interaction_type | pause                                
 watch_time_sec   | NULL                                 
 device_type      | mobile                               
-RECORD 1------------------------------------------------
 user_id          | 1c076e33-6d54-4adb-92c0-adcef5b2b305 
 video_id         | f9cbdb1f-44cd-40e5-9fa1-268c37e5bb22 
 event_id         | f8581bfd-774d-4c64-9805-741fc1c9a261 
 event_timestamp  | 2026-07-13T19:23:14.691925+00:00     
 interaction_type | play                                 
 watch_time_sec   | 88                                   
 device_type      | tv                                   
-RECORD 2------------------------------------------------
 user_id      

In [ ]:
# Missing Values

In [4]:
dfe.select([
    F.count(F.when(F.col(c).isNull(),
c)).alias(c)
    for c in dfe.columns
]).show(vertical=True)

-RECORD 0---------------
 user_id          | 0   
 video_id         | 0   
 event_id         | 0   
 event_timestamp  | 0   
 interaction_type | 0   
 watch_time_sec   | 510 
 device_type      | 0   



In [ ]:
# Uniqueness

In [6]:
dfe.groupBy("user_id").count().filter("count > 1").count()

251

In [7]:
dfe.groupBy("video_id").count().filter("count > 1").count()

12

In [8]:
dfe.groupBy("event_id").count().filter("count > 1").count()

0

In [12]:
dfe.groupBy("interaction_type").count().show()

+----------------+-----+
|interaction_type|count|
+----------------+-----+
|        complete|  253|
|           pause|  257|
|            play|  237|
|            like|  253|
+----------------+-----+



In [13]:
dfe.groupBy("device_type").count().show()

+-----------+-----+
|device_type|count|
+-----------+-----+
|        web|  325|
|         tv|  336|
|     mobile|  339|
+-----------+-----+



In [ ]:
# Cardinality

In [9]:
dfe.select("user_id").distinct().count()

632

In [10]:
dfe.select("video_id").distinct().count()

988

In [11]:
dfe.select("event_id").distinct().count()

1000

In [ ]:
# Business Logic & Consistency Checks 

In [14]:
dfe.groupBy("interaction_type").count().orderBy(F.desc("count")).show()

+----------------+-----+
|interaction_type|count|
+----------------+-----+
|           pause|  257|
|        complete|  253|
|            like|  253|
|            play|  237|
+----------------+-----+



In [16]:
dfe.groupBy("interaction_type") \
    .agg(
        F.count("*").alias("total"),
    F.count("watch_time_sec").alias("watch_time_count"),
    F.avg("watch_time_sec").alias("avg_watch_time"),
    F.min("watch_time_sec").alias("min_watch_time"),
    F.max("watch_time_sec").alias("max_watch_time")
        
    ) \
.orderBy(F.desc("total")) \
.show()

[Stage 53:=============================>                            (1 + 1) / 2]

+----------------+-----+----------------+-----------------+--------------+--------------+
|interaction_type|total|watch_time_count|   avg_watch_time|min_watch_time|max_watch_time|
+----------------+-----+----------------+-----------------+--------------+--------------+
|           pause|  257|               0|             NULL|          NULL|          NULL|
|        complete|  253|             253|62.88932806324111|            10|            99|
|            like|  253|               0|             NULL|          NULL|          NULL|
|            play|  237|             237|60.35443037974684|            10|            99|
+----------------+-----+----------------+-----------------+--------------+--------------+



In [17]:
dfe.groupBy("user_id").count().orderBy(F.desc("count")).show(20)

+--------------------+-----+
|             user_id|count|
+--------------------+-----+
|f1707fb6-eb55-4db...|    7|
|629410a5-5947-46b...|    5|
|c1fb824c-5996-42c...|    5|
|d28a3a9e-e34f-4e7...|    5|
|bf5094da-6767-4c5...|    5|
|4a622650-184a-400...|    4|
|5bd9869f-90f2-439...|    4|
|0dc7c887-fcd7-487...|    4|
|302642dc-3537-4be...|    4|
|4d7e3f88-2f11-47b...|    4|
|f5b08190-dff4-47b...|    4|
|f4ccffe5-cba4-45e...|    4|
|37268276-0b54-4f8...|    4|
|b3f10e29-eae4-47e...|    4|
|c0b0ad45-e4c8-447...|    4|
|633bcf1c-ab7b-41a...|    4|
|9c121d94-768c-461...|    4|
|14f54ee3-3cfd-47e...|    4|
|29cce4e9-069c-489...|    4|
|209e0a31-2b04-49c...|    4|
+--------------------+-----+
only showing top 20 rows



In [21]:
dfe.groupBy("video_id").count().orderBy(F.desc("count")).show(20)

+--------------------+-----+
|            video_id|count|
+--------------------+-----+
|d85d346a-9a45-4fb...|    2|
|6ae17766-42af-43a...|    2|
|7d0c4437-2d83-408...|    2|
|fe851b11-ff1f-420...|    2|
|0f97330e-8b69-4a0...|    2|
|493773ff-6b8f-4e5...|    2|
|1d3d4829-0729-409...|    2|
|5b972463-0d75-4d5...|    2|
|64f4a365-0197-440...|    2|
|c0bcdd3d-2b4f-4b4...|    2|
|f2310f90-b103-440...|    2|
|3efc8846-b27e-49b...|    2|
|d779ba5a-467a-473...|    1|
|a6bf5fa2-3ac0-4d6...|    1|
|f1aa4184-57e9-492...|    1|
|f0e3412e-5353-41f...|    1|
|57259ef8-0736-4f6...|    1|
|de75e845-33cd-41f...|    1|
|4cb67c95-100d-4e5...|    1|
|8534efc0-fd8e-4f5...|    1|
+--------------------+-----+
only showing top 20 rows



In [18]:
dfe.groupBy("device_type","interaction_type").count().orderBy("device_type",F.desc("count")).show()

+-----------+----------------+-----+
|device_type|interaction_type|count|
+-----------+----------------+-----+
|     mobile|           pause|   98|
|     mobile|            play|   87|
|     mobile|            like|   80|
|     mobile|        complete|   74|
|         tv|        complete|   90|
|         tv|            like|   86|
|         tv|           pause|   86|
|         tv|            play|   74|
|        web|        complete|   89|
|        web|            like|   87|
|        web|            play|   76|
|        web|           pause|   73|
+-----------+----------------+-----+



In [20]:
dfe.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
).groupBy(
    F.hour("event_timestamp").alias("hour")
).count() \
.orderBy("hour") \
.show()

+----+-----+
|hour|count|
+----+-----+
|  19| 1000|
+----+-----+



In [23]:
dfe.groupBy("interaction_type").agg(
    F.count("*").alias("total"),
    F.count("watch_time_sec").alias("watch_time_count"),
    F.sum(
        F.when(
            F.col("interaction_type").isin("play", "complete") &
            F.col("watch_time_sec").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_watch_time"),
    F.sum(
        F.when(
            F.col("interaction_type").isin("like", "pause") &
            F.col("watch_time_sec").isNotNull(),
            1
        ).otherwise(0)
    ).alias("unexpected_watch_time")
).show()

+----------------+-----+----------------+------------------+---------------------+
|interaction_type|total|watch_time_count|missing_watch_time|unexpected_watch_time|
+----------------+-----+----------------+------------------+---------------------+
|        complete|  253|             253|                 0|                    0|
|           pause|  257|               0|                 0|                    0|
|            play|  237|             237|                 0|                    0|
|            like|  253|               0|                 0|                    0|
+----------------+-----+----------------+------------------+---------------------+



In [25]:
dfe.filter(
    F.col("watch_time_sec").isNotNull() &
    ((F.col("watch_time_sec") <= 0) | (F.col("watch_time_sec") > 120))
).count()

0

In [26]:
dfe.filter(
    ~F.col("interaction_type").isin("like", "play", "complete", "pause")
).count()

0

In [27]:
dfe.filter(
    ~F.col("device_type").isin("mobile", "tv", "web")
).count()

0

In [29]:
dfe.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
).filter(
    F.col("event_timestamp").isNull()
).count()

0